## Generating Ground Truth Data

In [51]:
import os
from dotenv import load_dotenv
load_dotenv(dotenv_path="../../01_module_agentic_rag/.env")
from openai import OpenAI

openai_client = OpenAI(
    api_key=os.getenv("LMSTUDIO_API_KEY"),
    base_url=os.getenv("LMSTUDIO_HOST")
)

In [ ]:
model = "qwen/qwen3.5-9b"

In [53]:
from ingest import load_faq_data
documents = load_faq_data()

In [54]:
documents[0]

{'id': '0e38656cfb',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'How do I submit homework?',
 'answer': "- Do the tasks locally\n- Publish your code (e.g., in your own GitHub repo)\n- Submit your answers via the homework form and include the URL to your code\n- You will see the answers only after the deadline\n- Homeworks are in the cohorts folder, e.g. for 2025 it's [`cohorts/2025`](https://github.com/DataTalksClub/machine-learning-zoomcamp/tree/master/cohorts/2025)\n- The forms for submitting the homework are in the [course management platform](https://courses.datatalks.club/)"}

In [55]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

118

In [56]:
# assign llm course to documents var
documents = documents_llm

In [57]:
len(documents)

118

In [58]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [59]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.

Use casual language
""".strip()

In [60]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [61]:
import json

user_prompt = json.dumps(doc)

In [62]:
user_prompt

'{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'

In [63]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [64]:
response = openai_client.chat.completions.parse(
    model=model,
    messages=messages,
    response_format=Questions
)

KeyboardInterrupt: 

In [ ]:
response

ParsedChatCompletion[TypeVar](id='chatcmpl-oJD45SBv2znlDd6BhtAL16oe6QsLm2yg', choices=[ParsedChoice[TypeVar](finish_reason='stop', index=0, logprobs=None, message=ParsedChatCompletionMessage[TypeVar](content='{\n  "questions": [\n    "Q: Hey, did I miss out if I found this late?\\nA: Enrollment is open, but the badge requires sending your work before we stop taking them in.",\n    "Q: Will I still be eligible for the cert if I start later?\\nA: That\'s possible, just ensure you hand over your deliverable while they are currently open for entries.",\n    "Q: Does starting now affect getting the badge?\\nA: It depends on whether you upload your task during the active intake phase.",\n    "Q: Is there a cutoff for new students wanting the cert?\\nA: It works if you manage to complete the task while we\'re currently open for entries.",\n    "Q: Is there a chance to finish late and get the proof?\\nA: You can do that as long as you hand in your final task before the intake period ends."\n  

In [ ]:
results = response.choices[0].message.parsed
results

Questions(questions=['Q: Hey, did I miss out if I found this late?\nA: Enrollment is open, but the badge requires sending your work before we stop taking them in.', "Q: Will I still be eligible for the cert if I start later?\nA: That's possible, just ensure you hand over your deliverable while they are currently open for entries.", 'Q: Does starting now affect getting the badge?\nA: It depends on whether you upload your task during the active intake phase.', "Q: Is there a cutoff for new students wanting the cert?\nA: It works if you manage to complete the task while we're currently open for entries.", 'Q: Is there a chance to finish late and get the proof?\nA: You can do that as long as you hand in your final task before the intake period ends.'])

In [ ]:
results.questions

['Q: Hey, did I miss out if I found this late?\nA: Enrollment is open, but the badge requires sending your work before we stop taking them in.',
 "Q: Will I still be eligible for the cert if I start later?\nA: That's possible, just ensure you hand over your deliverable while they are currently open for entries.",
 'Q: Does starting now affect getting the badge?\nA: It depends on whether you upload your task during the active intake phase.',
 "Q: Is there a cutoff for new students wanting the cert?\nA: It works if you manage to complete the task while we're currently open for entries.",
 'Q: Is there a chance to finish late and get the proof?\nA: You can do that as long as you hand in your final task before the intake period ends.']

In [ ]:
usage_dict = {
    "prompt_tokens": response.usage.prompt_tokens,
    "total_tokens": response.usage.total_tokens
}

In [ ]:
usage_dict['prompt_tokens'], usage_dict['total_tokens']

(189, 7792)

In [ ]:
from evaluation_utils import llm_structured_chat_completions, calc_price_chat_completions

ImportError: cannot import name 'calc_price_chat_completions' from 'evaluation_utils' (/home/user1129/MyDocuments/10_14_Life_Admin/13_Technology/100_Coding/100_01_Python/llm_ai_engineering/llm_zoomcamp_portfolio/modules/04_evaluation/notebooks/evaluation_utils.py)

In [ ]:
result, usage = llm_structured_chat_completions(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

In [ ]:
cost = calc_price_chat_completions(usage)

cost

AttributeError: 'CompletionUsage' object has no attribute 'input_tokens'

In [ ]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'Is it actually too late to get on board right now?',
  'document': '74eb249bbf'},
 {'question': 'Will starting late mean I miss out on the badge?',
  'document': '74eb249bbf'},
 {'question': 'Do I need to finish the final task before a specific deadline?',
  'document': '74eb249bbf'},
 {'question': 'Can new people participate even after the initial sign up period has passed?',
  'document': '74eb249bbf'},
 {'question': 'How does being enrolled late affect getting the award?',
  'document': '74eb249bbf'}]

In [ ]:
from evaluation_utils import llm_structured_retry_chat_completions

In [ ]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry_chat_completions(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [ ]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]